# 📊 Comparaison multi-tags — Fink/LSST

Ce notebook récupère **tous les tags disponibles** via l'API Fink/LSST et produit pour chacun :
- Nombre d'alertes et d'objets uniques
- Distribution par filtre LSST
- Distribution en flux PSF (nJy)
- Distribution spatiale (RA, Dec)

Une figure de synthèse compare tous les tags côte à côte.

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/tags` · `/api/v1/sources`

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
N_PER_TAG  = 200          # Nombre max d'alertes récupérées par tag
STARTDATE  = None         # "2026-01-01 00:00:00" ou None
STOPDATE   = None         # "2026-03-01 00:00:00" ou None
BASE_URL   = "https://api.lsst.fink-portal.org"
SAVE_FIGS  = True
OUTPUT_DIR = "multitag_outputs"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import display as ipy_display, HTML
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

mpl.rcParams.update({
    'font.size'        : 11,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'legend.fontsize'  : 9,
    'figure.titlesize' : 14,
    'figure.titleweight': 'bold',
})

FILTER_COLORS = {
    'u': '#7B2FBE', 'g': '#0077BB', 'r': '#33AA77',
    'i': '#DDAA33', 'z': '#BB5500', 'y': '#AA0000',
}
# Couleur par tag (palette distincte)
TAG_PALETTE = [
    '#4477AA', '#EE6677', '#228833', '#CCBB44',
    '#66CCEE', '#AA3377', '#BBBBBB',
]

MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')

print("Imports OK ✓")

## 3. Récupération de la liste des tags

In [ ]:
r = requests.get(f"{BASE_URL}/api/v1/tags")
r.raise_for_status()
TAGS_DOC = r.json()   # dict {tag_name: description}
ALL_TAGS = list(TAGS_DOC.keys())

print(f"Tags disponibles ({len(ALL_TAGS)}) :")
for tag, doc in TAGS_DOC.items():
    print(f"  {tag:45s} {doc}")

## 4. Téléchargement des alertes pour chaque tag

In [ ]:
def fetch_tag_data(tag, n, startdate=None, stopdate=None):
    """
    Récupère les N dernières alertes d'un tag avec colonnes
    diaObjectId, ra, dec, band, psfFlux, midpointMjdTai.
    """
    payload = {
        "tag"          : tag,
        "n"            : n,
        "output-format": "json",
    }
    if startdate:
        payload["startdate"] = startdate
    if stopdate:
        payload["stopdate"] = stopdate
    resp = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    # Normaliser les noms de colonnes (supprimer préfixe r:)
    df.columns = [c.replace('r:', '') for c in df.columns]
    for col in ['psfFlux', 'ra', 'dec', 'midpointMjdTai']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


tag_data = {}   # {tag: DataFrame}

for tag in tqdm(ALL_TAGS, desc='Téléchargement tags', unit='tag'):
    try:
        df = fetch_tag_data(tag, N_PER_TAG, STARTDATE, STOPDATE)
        tag_data[tag] = df
        id_col = next((c for c in df.columns if 'diaObjectId' in c), None)
        n_obj  = df[id_col].nunique() if id_col else '?'
        print(f"  ✓  {tag:45s} {len(df):5d} alertes  {n_obj:>4} objets uniques")
    except Exception as e:
        print(f"  ✗  {tag} — erreur : {e}")
        tag_data[tag] = pd.DataFrame()

print(f"\n✓ {len(tag_data)} tags chargés.")

## 5. Tableau récapitulatif

In [ ]:
rows = []
for tag, df in tag_data.items():
    if df.empty:
        rows.append({'Tag': tag, 'N alertes': 0, 'N objets': 0,
                     'Filtres': '—', 'Flux médian (nJy)': np.nan,
                     'RA médian': np.nan, 'Dec médian': np.nan})
        continue
    id_col  = next((c for c in df.columns if 'diaObjectId' in c), None)
    n_obj   = df[id_col].nunique() if id_col else np.nan
    filtres = ', '.join(sorted(df['band'].dropna().unique())) if 'band' in df.columns else '—'
    rows.append({
        'Tag'              : tag,
        'Description'      : TAGS_DOC.get(tag, ''),
        'N alertes'        : len(df),
        'N objets'         : int(n_obj) if not np.isnan(n_obj) else 0,
        'Filtres'          : filtres,
        'Flux médian (nJy)': round(df['psfFlux'].median(), 1) if 'psfFlux' in df.columns else np.nan,
        'RA médian'        : round(df['ra'].median(), 2) if 'ra' in df.columns else np.nan,
        'Dec médian'       : round(df['dec'].median(), 2) if 'dec' in df.columns else np.nan,
    })

df_stats = pd.DataFrame(rows)

html = df_stats.to_html(index=False, escape=False, border=0, classes='fink-table')
style = """
<style>
  .fink-table { border-collapse: collapse; font-size: 12px; width: 100%; }
  .fink-table th { background: #f0f0f0; padding: 6px 10px;
                   border-bottom: 2px solid #ccc; text-align: left; }
  .fink-table td { padding: 4px 10px; border-bottom: 1px solid #eee; text-align: left; }
  .fink-table tr:hover td { background: #fafafa; }
</style>
"""
ipy_display(HTML(style + html))

## 6. Figure de synthèse — comparaison multi-tags

4 panneaux par tag : distribution filtres · distribution flux · RA · Dec

In [ ]:
n_tags  = len([t for t, df in tag_data.items() if not df.empty])
tags_ok = [t for t, df in tag_data.items() if not df.empty]

fig, axes = plt.subplots(
    n_tags, 4,
    figsize=(18, 3.5 * n_tags),
    layout='constrained'
)
if n_tags == 1:
    axes = axes[np.newaxis, :]

fig.suptitle("Comparaison multi-tags — Fink/LSST", y=1.01)

for row_idx, tag in enumerate(tags_ok):
    df    = tag_data[tag]
    color = TAG_PALETTE[row_idx % len(TAG_PALETTE)]
    axes[row_idx, 0].set_ylabel(tag, fontsize=10, fontweight='bold',
                                 rotation=90, labelpad=8)

    # ── Col 0 : distribution filtres ──────────────────────────────────────
    ax = axes[row_idx, 0]
    if 'band' in df.columns:
        vc = df['band'].value_counts().sort_index()
        bar_colors = [FILTER_COLORS.get(b, '#888') for b in vc.index]
        ax.bar(vc.index, vc.values, color=bar_colors, edgecolor='white', linewidth=0.5)
        ax.set_xlabel('Filtre')
        ax.set_ylabel('N alertes')
    if row_idx == 0:
        ax.set_title('Distribution filtres')
    ax.grid(True, axis='y', alpha=0.25, linewidth=0.5)

    # ── Col 1 : distribution flux PSF ─────────────────────────────────────
    ax = axes[row_idx, 1]
    if 'psfFlux' in df.columns:
        flux = df['psfFlux'].dropna()
        flux_pos = flux[flux > 0]
        if not flux_pos.empty:
            ax.hist(np.log10(flux_pos), bins=30, color=color,
                    edgecolor='white', linewidth=0.4, alpha=0.85)
            ax.set_xlabel('log₁₀ Flux PSF (nJy)')
            ax.set_ylabel('N alertes')
    if row_idx == 0:
        ax.set_title('Distribution flux (log)')
    ax.grid(True, axis='y', alpha=0.25, linewidth=0.5)

    # ── Col 2 : distribution RA ────────────────────────────────────────────
    ax = axes[row_idx, 2]
    if 'ra' in df.columns:
        ax.hist(df['ra'].dropna(), bins=36, range=(0, 360),
                color=color, edgecolor='white', linewidth=0.4, alpha=0.85)
        ax.set_xlabel('RA (deg)')
        ax.set_ylabel('N alertes')
        ax.set_xlim(0, 360)
    if row_idx == 0:
        ax.set_title('Distribution RA')
    ax.grid(True, axis='y', alpha=0.25, linewidth=0.5)

    # ── Col 3 : distribution Dec ───────────────────────────────────────────
    ax = axes[row_idx, 3]
    if 'dec' in df.columns:
        ax.hist(df['dec'].dropna(), bins=30, range=(-90, 10),
                color=color, edgecolor='white', linewidth=0.4, alpha=0.85)
        ax.set_xlabel('Dec (deg)')
        ax.set_ylabel('N alertes')
        ax.set_xlim(-90, 10)
        ax.axvline(-62, color='goldenrod', lw=1.0, ls='--', alpha=0.7,
                   label='Limite LSST')
        if row_idx == 0:
            ax.legend(fontsize=8)
    if row_idx == 0:
        ax.set_title('Distribution Dec')
    ax.grid(True, axis='y', alpha=0.25, linewidth=0.5)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/multitag_stats.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/multitag_stats.png", bbox_inches='tight', dpi=150)
    print(f"✓ Sauvegardé : {OUTPUT_DIR}/multitag_stats.pdf / .png")

plt.show()

## 7. Recouvrement entre tags

Combien d'objets apparaissent dans plusieurs tags simultanément ?

In [ ]:
# Ensembles de diaObjectId par tag
sets = {}
for tag, df in tag_data.items():
    if df.empty:
        continue
    id_col = next((c for c in df.columns if 'diaObjectId' in c), None)
    if id_col:
        sets[tag] = set(df[id_col].astype(str).unique())

tags_with_data = list(sets.keys())
n = len(tags_with_data)

# Matrice de recouvrement
matrix = np.zeros((n, n), dtype=int)
for i, t1 in enumerate(tags_with_data):
    for j, t2 in enumerate(tags_with_data):
        matrix[i, j] = len(sets[t1] & sets[t2])

fig, ax = plt.subplots(figsize=(max(6, n * 1.4), max(5, n * 1.2)),
                        layout='constrained')
fig.suptitle("Recouvrement entre tags (N objets communs)")

im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto')
fig.colorbar(im, ax=ax, label='N objets communs')

short_names = [t.replace('_candidate','').replace('extragalactic_','ext_') for t in tags_with_data]
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(short_names, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(short_names, fontsize=9)

for i in range(n):
    for j in range(n):
        ax.text(j, i, str(matrix[i, j]), ha='center', va='center',
                fontsize=9, color='black' if matrix[i,j] < matrix.max()*0.6 else 'white')

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/multitag_overlap.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/multitag_overlap.png", bbox_inches='tight', dpi=150)
    print(f"✓ Sauvegardé : {OUTPUT_DIR}/multitag_overlap.pdf / .png")

plt.show()

## 8. Carte Mollweide — tous tags superposés

In [ ]:
from astropy.coordinates import SkyCoord, BarycentricTrueEcliptic
import astropy.units as u

def radec_to_mollweide(ra_deg, dec_deg):
    ra  = np.asarray(ra_deg,  dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    return -np.deg2rad(ra - 180.0), np.deg2rad(dec)

def split_curve(x, y, threshold=np.pi * 0.8):
    breaks = np.where(np.abs(np.diff(x)) > threshold)[0] + 1
    return np.split(x, breaks), np.split(y, breaks)

# Plan galactique
l = np.linspace(0, 360, 1000)
gc = SkyCoord(l=l*u.deg, b=np.zeros(1000)*u.deg, frame='galactic').icrs
idx = np.argsort(gc.ra.deg)
gal_x, gal_y = radec_to_mollweide(gc.ra.deg[idx], gc.dec.deg[idx])

fig = plt.figure(figsize=(16, 8), layout='constrained')
ax  = fig.add_subplot(111, projection='mollweide')
fig.suptitle("Carte céleste — tous tags Fink/LSST")
ax.grid(True, color='gray', alpha=0.3, linewidth=0.5, linestyle='--')

# Plan galactique
for sx, sy in zip(*split_curve(gal_x, gal_y)):
    ax.plot(sx, sy, color='goldenrod', lw=1.2,
            label='Plan galactique' if sx is split_curve(gal_x, gal_y)[0][0] else '')

handles = [plt.Line2D([0],[0], color='goldenrod', lw=1.2, label='Plan galactique')]

for idx_t, (tag, df) in enumerate(tag_data.items()):
    if df.empty or 'ra' not in df.columns or 'dec' not in df.columns:
        continue
    sub = df.dropna(subset=['ra', 'dec'])
    px, py = radec_to_mollweide(sub['ra'].values, sub['dec'].values)
    color  = TAG_PALETTE[idx_t % len(TAG_PALETTE)]
    short  = tag.replace('_candidate','').replace('extragalactic_','ext_')
    ax.scatter(px, py, s=8, color=color, alpha=0.6, zorder=5,
               edgecolors='none', label=short)
    handles.append(plt.Line2D([0],[0], marker='o', color='w',
                               markerfacecolor=color, markersize=7, label=short))

ax.legend(handles=handles, loc='lower left', fontsize=8,
          framealpha=0.75, ncol=2)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/multitag_skymap.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/multitag_skymap.png", bbox_inches='tight', dpi=150)
    print(f"✓ Sauvegardé : {OUTPUT_DIR}/multitag_skymap.pdf / .png")

plt.show()